# LeetCode 1030 — Matrix Cells in Distance Order bằng Python

**Link đề:** https://leetcode.com/problems/matrix-cells-in-distance-order/
**Độ khó:** Easy · **Chủ đề:** Array, Math, Geometry, Sorting, Matrix

```python
class Solution:
    def allCellsDistOrder(self, rows: int, cols: int, rCenter: int, cCenter: int) -> list[list[int]]:
        ...
```

## Mục tiêu học
1. Hiểu **khoảng cách Manhattan** `|r1 - r2| + |c1 - c2|` trên lưới ô vuông.
2. Ôn công cụ: list comprehension lồng nhau, `abs`, `sorted(key=lambda ...)`, bucket (list các list), `deque` cho BFS.
3. Viết lời giải từ dễ đến tinh: sinh mọi ô rồi **sắp xếp** → **bucket sort** theo khoảng cách → **BFS** từ tâm → duyệt **vòng kim cương**.
4. Viết **bộ kiểm tra** cho bài có nhiều đáp án đúng (không so sánh bằng `==` với một đáp án cố định).
5. Trace, chứng minh, phân tích Time/Space, so sánh 4 cách.
6. Làm bài tập biến thể: khoảng cách Chebyshev, chỉ lấy ô ở khoảng cách `d`, k ô gần nhất.

**Cách học:** đọc markdown → đoán output → chạy cell → so sánh. Các cell TODO chỉ để luyện, không tự chạy.

## 1. Đề bài

Có một ma trận `rows × cols`, ta đứng ở ô `(rCenter, cCenter)`. Trả về tọa độ **mọi ô** trong ma trận, sắp theo khoảng cách tới `(rCenter, cCenter)` **từ nhỏ đến lớn**. Các ô cùng khoảng cách được phép ở **thứ tự bất kỳ**.

Khoảng cách giữa `(r1, c1)` và `(r2, c2)` là `|r1 - r2| + |c1 - c2|` (khoảng cách **Manhattan**).

| rows, cols | tâm | Một output hợp lệ | Khoảng cách tương ứng |
|---|---|---|---|
| 1, 2 | (0, 0) | `[[0,0],[0,1]]` | `[0,1]` |
| 2, 2 | (0, 1) | `[[0,1],[0,0],[1,1],[1,0]]` | `[0,1,1,2]` |
| 2, 3 | (1, 2) | `[[1,2],[0,2],[1,1],[0,1],[1,0],[0,0]]` | `[0,1,1,2,2,3]` |

Ví dụ 2 cũng chấp nhận `[[0,1],[1,1],[0,0],[1,0]]` (đổi chỗ hai ô cùng khoảng cách 1).

**Constraints:** `1 <= rows, cols <= 100`, `0 <= rCenter < rows`, `0 <= cCenter < cols`.

Câu hỏi tự đặt:
- Có bao nhiêu ô? (`rows · cols`, tối đa 10 000.)
- Khoảng cách lớn nhất có thể là bao nhiêu? (`(rows - 1) + (cols - 1)`, tối đa 198.)
- Vì đáp án không duy nhất, **test** thế nào? (Mục 10.)

In [1]:
def manhattan(r1, c1, r2, c2):
    return abs(r1 - r2) + abs(c1 - c2)

out = [[1, 2], [0, 2], [1, 1], [0, 1], [1, 0], [0, 0]]
print([manhattan(r, c, 1, 2) for r, c in out])

[0, 1, 1, 2, 2, 3]


`for r, c in out` **unpack** mỗi phần tử `[r, c]` thành hai biến.

## 2. Nền tảng

### 2.1. Khoảng cách Manhattan — hình dung

Đi trong thành phố ô bàn cờ (như Manhattan, New York) chỉ được đi ngang/dọc: số bước ít nhất là `|Δhàng| + |Δcột|`. Các ô cùng khoảng cách `d` tạo thành **hình thoi (kim cương)** quanh tâm.

In [2]:
rows, cols, rc, cc = 5, 7, 2, 3
for r in range(rows):
    print(" ".join(str(manhattan(r, c, rc, cc)) for c in range(cols)))

5 4 3 2 3 4 5
4 3 2 1 2 3 4
3 2 1 0 1 2 3
4 3 2 1 2 3 4
5 4 3 2 3 4 5


Nhìn bảng: ô tâm là `0`, các ô `1` bao quanh thành hình thoi nhỏ, rồi `2`, `3`, ... lan ra như sóng.

### 2.2. Sinh mọi ô bằng list comprehension lồng nhau

In [3]:
cells = [[r, c] for r in range(2) for c in range(3)]
print(cells)

[[0, 0], [0, 1], [0, 2], [1, 0], [1, 1], [1, 2]]


Đọc từ trái sang phải giống hai vòng `for` lồng nhau: `for r ...` ở ngoài, `for c ...` ở trong.

### 2.3. `sorted(..., key=lambda ...)`

`lambda cell: ...` là **hàm vô danh** một dòng. `sorted(cells, key=f)` sắp các phần tử theo giá trị `f(cell)`. Python dùng **Timsort ổn định (stable)**: các phần tử có cùng khóa giữ nguyên thứ tự ban đầu.

In [4]:
square = lambda x: x * x
print(square(5))
words = ["ccc", "a", "bb", "dd"]
print(sorted(words, key=len))        # 'bb' trước 'dd' vì ổn định

25
['a', 'bb', 'dd', 'ccc']


### 2.4. Bucket — "list các list"

Khi khóa sắp xếp là số nguyên nhỏ (`0..D`), ta tạo `D + 1` "thùng", bỏ mỗi phần tử vào thùng của khóa nó, rồi nối các thùng theo thứ tự. Không cần so sánh → O(N + D).

**Cẩn thận:** `[[]] * k` tạo `k` tham chiếu tới **cùng một** list! Phải dùng `[[] for _ in range(k)]`.

In [5]:
wrong = [[]] * 3
wrong[0].append("x")
print("[[]] * 3 sau append:", wrong)
right = [[] for _ in range(3)]
right[0].append("x")
print("[[] for _ in range(3)]:", right)

[[]] * 3 sau append: [['x'], ['x'], ['x']]
[[] for _ in range(3)]: [['x'], [], []]


### 2.5. `deque` và BFS (Breadth-First Search — duyệt theo chiều rộng)

BFS từ một ô: dùng hàng đợi; lấy ô ở đầu, đẩy các ô kề chưa thăm vào cuối. Trên lưới không vật cản, BFS thăm các ô **theo thứ tự số bước tăng dần** — đúng bằng khoảng cách Manhattan.

In [6]:
from collections import deque
q = deque()
q.append((0, 0))
q.append((0, 1))
print("popleft:", q.popleft(), "| còn:", list(q))
DIRS = [(1, 0), (-1, 0), (0, 1), (0, -1)]
print("4 hướng kề:", DIRS)

popleft: (0, 0) | còn: [(0, 1)]
4 hướng kề: [(1, 0), (-1, 0), (0, 1), (0, -1)]


## 3. Cách trực giác — sinh mọi ô rồi sắp xếp

### 3.1. Cách 1: sort theo khoảng cách

Đúng như đề nói: liệt kê hết, rồi sắp theo khoảng cách.

In [7]:
def all_cells_sort(rows, cols, rCenter, cCenter):
    cells = [[r, c] for r in range(rows) for c in range(cols)]
    return sorted(cells, key=lambda cell: abs(cell[0] - rCenter) + abs(cell[1] - cCenter))

print(all_cells_sort(1, 2, 0, 0))
print(all_cells_sort(2, 2, 0, 1))
print(all_cells_sort(2, 3, 1, 2))

[[0, 0], [0, 1]]
[[0, 1], [0, 0], [1, 1], [1, 0]]
[[1, 2], [0, 2], [1, 1], [0, 1], [1, 0], [0, 0]]


Output ví dụ 3 khác thứ tự trong đề nhưng **vẫn hợp lệ** (các ô cùng khoảng cách đổi chỗ). Vì sort ổn định, các ô cùng khoảng cách giữ thứ tự sinh ra (theo hàng rồi cột).

**Độ phức tạp:** `N = rows · cols` ô; sort O(N log N). Với N = 10 000 vẫn rất nhanh. Đây là lời giải hoàn toàn chấp nhận được trên LeetCode.

### 3.2. Viết bằng `list.sort` tại chỗ

In [8]:
def all_cells_sort_inplace(rows, cols, rCenter, cCenter):
    cells = [[r, c] for r in range(rows) for c in range(cols)]
    cells.sort(key=lambda rc: abs(rc[0] - rCenter) + abs(rc[1] - cCenter))
    return cells

print(all_cells_sort_inplace(2, 3, 1, 2))

[[1, 2], [0, 2], [1, 1], [0, 1], [1, 0], [0, 0]]


## 4. Nút thắt & insight

Sort tốn O(N log N) vì phải **so sánh** các khóa. Nhưng khóa ở đây đặc biệt:
- là **số nguyên**,
- nằm trong khoảng **nhỏ** `0..D` với `D = (rows - 1) + (cols - 1) ≤ 198`.

⇒ Không cần so sánh: dùng **bucket sort** (counting sort) theo khoảng cách → O(N + D).

Một góc nhìn khác: khoảng cách Manhattan chính là **số bước BFS** trên lưới → BFS từ tâm cho thứ tự đúng một cách tự nhiên, O(N).

Góc nhìn thứ ba (hình học): ô ở khoảng cách `d` nằm trên **viền hình thoi** bán kính `d` → duyệt `d = 0, 1, 2, ...` và liệt kê viền, bỏ ô ngoài ma trận.

In [9]:
rows, cols, rc, cc = 2, 3, 1, 2
D = (rows - 1) + (cols - 1)
print("Khoảng cách lớn nhất có thể (giới hạn trên):", D)
print("Khoảng cách xa nhất thật sự từ tâm:", max(rc, rows - 1 - rc) + max(cc, cols - 1 - cc))

Khoảng cách lớn nhất có thể (giới hạn trên): 3
Khoảng cách xa nhất thật sự từ tâm: 3


Khoảng cách xa nhất thật sự là `max(rc, rows-1-rc) + max(cc, cols-1-cc)` (đi về góc xa nhất). Dùng giá trị này làm số thùng sẽ không có thùng thừa ở cuối — nhưng dùng `D` cũng đúng (thùng thừa chỉ rỗng).

## 5. Lời giải chính — bucket sort theo khoảng cách

```python
max_d = max(rCenter, rows-1-rCenter) + max(cCenter, cols-1-cCenter)   # Đoạn 1
buckets = [[] for _ in range(max_d + 1)]                              # Đoạn 2
for r in range(rows):                                                 # Đoạn 3: rải ô vào thùng
    for c in range(cols):
        buckets[abs(r - rCenter) + abs(c - cCenter)].append([r, c])
result = []                                                           # Đoạn 4: nối các thùng
for bucket in buckets:
    result.extend(bucket)
return result
```

`result.extend(bucket)` thêm **từng phần tử** của `bucket` vào `result` (khác `append(bucket)` sẽ thêm cả list như một phần tử).

In [10]:
a = [1]
a.extend([2, 3])
bb = [1]
bb.append([2, 3])
print("extend:", a, "| append:", bb)

extend: [1, 2, 3] | append: [1, [2, 3]]


In [11]:
def all_cells_bucket(rows, cols, rCenter, cCenter):
    max_d = max(rCenter, rows - 1 - rCenter) + max(cCenter, cols - 1 - cCenter)
    buckets = [[] for _ in range(max_d + 1)]
    for r in range(rows):
        for c in range(cols):
            buckets[abs(r - rCenter) + abs(c - cCenter)].append([r, c])
    result = []
    for bucket in buckets:
        result.extend(bucket)
    return result

print(all_cells_bucket(1, 2, 0, 0))
print(all_cells_bucket(2, 2, 0, 1))
print(all_cells_bucket(2, 3, 1, 2))

[[0, 0], [0, 1]]
[[0, 1], [0, 0], [1, 1], [1, 0]]
[[1, 2], [0, 2], [1, 1], [0, 1], [1, 0], [0, 0]]


### 5.1. Cách 3 — BFS từ tâm

In [12]:
def all_cells_bfs(rows, cols, rCenter, cCenter):
    visited = [[False] * cols for _ in range(rows)]
    visited[rCenter][cCenter] = True
    queue = deque([(rCenter, cCenter)])
    result = []
    while queue:
        r, c = queue.popleft()
        result.append([r, c])
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols and not visited[nr][nc]:
                visited[nr][nc] = True          # đánh dấu khi ĐẨY vào, không phải khi lấy ra
                queue.append((nr, nc))
    return result

print(all_cells_bfs(2, 3, 1, 2))

[[1, 2], [0, 2], [1, 1], [0, 1], [1, 0], [0, 0]]


- `[[False] * cols for _ in range(rows)]` tạo ma trận `rows × cols` (mỗi hàng là list **riêng** — xem bẫy ở mục 2.4).
- `0 <= nr < rows` là so sánh chuỗi: `0 <= nr and nr < rows` → ô nằm trong ma trận.
- Đánh dấu `visited` **khi đẩy vào queue** để một ô không bị đẩy nhiều lần.

### 5.2. Cách 4 — Duyệt viền hình thoi theo từng `d`

Với mỗi `d`, các ô có `|dr| + |dc| = d`. Duyệt `dr` từ `-d` tới `d`; khi đó `|dc| = d - |dr|`, cho tối đa 2 giá trị `dc = ±(d - |dr|)` (1 giá trị nếu bằng 0). Giữ những ô nằm trong ma trận.

In [13]:
def all_cells_diamond(rows, cols, rCenter, cCenter):
    max_d = max(rCenter, rows - 1 - rCenter) + max(cCenter, cols - 1 - cCenter)
    result = []
    for d in range(max_d + 1):
        for dr in range(-d, d + 1):
            r = rCenter + dr
            if not 0 <= r < rows:
                continue
            k = d - abs(dr)
            for c in {cCenter - k, cCenter + k}:      # set: tránh trùng khi k = 0
                if 0 <= c < cols:
                    result.append([r, c])
    return result

print(all_cells_diamond(2, 3, 1, 2))

[[1, 2], [0, 2], [1, 1], [0, 1], [1, 0], [0, 0]]


`{a, b}` là **set** (tập hợp): nếu `a == b` (khi `k = 0`) thì chỉ còn một phần tử → không thêm trùng ô.

## 6. TRACE từng bước

### 6.1. Trace bucket sort với `rows = 2, cols = 3`, tâm `(1, 2)`

| (r, c) | `|r-1|` | `|c-2|` | d | thùng sau khi thêm |
|---|:-:|:-:|:-:|---|
| (0,0) | 1 | 2 | 3 | B3 = [(0,0)] |
| (0,1) | 1 | 1 | 2 | B2 = [(0,1)] |
| (0,2) | 1 | 0 | 1 | B1 = [(0,2)] |
| (1,0) | 0 | 2 | 2 | B2 = [(0,1),(1,0)] |
| (1,1) | 0 | 1 | 1 | B1 = [(0,2),(1,1)] |
| (1,2) | 0 | 0 | 0 | B0 = [(1,2)] |

Nối `B0 + B1 + B2 + B3` → `[(1,2),(0,2),(1,1),(0,1),(1,0),(0,0)]` — trùng đúng output mẫu của đề.

In [14]:
def trace_bucket(rows, cols, rCenter, cCenter):
    max_d = max(rCenter, rows - 1 - rCenter) + max(cCenter, cols - 1 - cCenter)
    buckets = [[] for _ in range(max_d + 1)]
    for r in range(rows):
        for c in range(cols):
            d = abs(r - rCenter) + abs(c - cCenter)
            buckets[d].append([r, c])
            print(f"ô ({r},{c}): d = |{r}-{rCenter}| + |{c}-{cCenter}| = {d} -> B{d} = {buckets[d]}")
    result = []
    for d, bucket in enumerate(buckets):
        result.extend(bucket)
        print(f"nối B{d}: result = {result}")
    return result

trace_bucket(2, 3, 1, 2)

ô (0,0): d = |0-1| + |0-2| = 3 -> B3 = [[0, 0]]
ô (0,1): d = |0-1| + |1-2| = 2 -> B2 = [[0, 1]]
ô (0,2): d = |0-1| + |2-2| = 1 -> B1 = [[0, 2]]
ô (1,0): d = |1-1| + |0-2| = 2 -> B2 = [[0, 1], [1, 0]]
ô (1,1): d = |1-1| + |1-2| = 1 -> B1 = [[0, 2], [1, 1]]
ô (1,2): d = |1-1| + |2-2| = 0 -> B0 = [[1, 2]]
nối B0: result = [[1, 2]]
nối B1: result = [[1, 2], [0, 2], [1, 1]]
nối B2: result = [[1, 2], [0, 2], [1, 1], [0, 1], [1, 0]]
nối B3: result = [[1, 2], [0, 2], [1, 1], [0, 1], [1, 0], [0, 0]]


[[1, 2], [0, 2], [1, 1], [0, 1], [1, 0], [0, 0]]

### 6.2. Trace BFS — xem "sóng" lan từ tâm

In thứ tự thăm (số thứ tự) lên lưới `4 × 5`, tâm `(1, 2)`: các số nhỏ ở gần tâm.

In [15]:
def trace_bfs_grid(rows, cols, rCenter, cCenter):
    order = all_cells_bfs(rows, cols, rCenter, cCenter)
    step = [[0] * cols for _ in range(rows)]
    for idx, (r, c) in enumerate(order):
        step[r][c] = idx
    print("Thứ tự thăm BFS:")
    for row in step:
        print(" ".join(f"{v:>2}" for v in row))
    print("Khoảng cách theo thứ tự thăm:", [abs(r - rCenter) + abs(c - cCenter) for r, c in order])

trace_bfs_grid(4, 5, 1, 2)

Thứ tự thăm BFS:
17  9  2  8 16
11  4  0  3 10
15  7  1  6 14
19 13  5 12 18
Khoảng cách theo thứ tự thăm: [0, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 4, 4]


`for idx, (r, c) in enumerate(order)` — unpack lồng: `enumerate` cho `(idx, [r, c])`, và `(r, c)` tách tiếp phần tử thứ hai.

Dãy khoảng cách theo thứ tự thăm là **không giảm** — đúng điều đề yêu cầu.

## 7. Code nộp LeetCode (Python 3)

In [16]:
class Solution:
    def allCellsDistOrder(self, rows: int, cols: int, rCenter: int, cCenter: int) -> list[list[int]]:
        max_d = max(rCenter, rows - 1 - rCenter) + max(cCenter, cols - 1 - cCenter)
        buckets = [[] for _ in range(max_d + 1)]
        for r in range(rows):
            for c in range(cols):
                buckets[abs(r - rCenter) + abs(c - cCenter)].append([r, c])
        result = []
        for bucket in buckets:
            result.extend(bucket)
        return result

Bản ngắn gọn bằng sort (cũng được chấp nhận):

In [17]:
class SolutionSort:
    def allCellsDistOrder(self, rows: int, cols: int, rCenter: int, cCenter: int) -> list[list[int]]:
        cells = [[r, c] for r in range(rows) for c in range(cols)]
        cells.sort(key=lambda rc: abs(rc[0] - rCenter) + abs(rc[1] - cCenter))
        return cells

## 8. Vì sao đúng?

**Bucket sort:**
- Mỗi ô `(r, c)` được thêm vào **đúng một** thùng — thùng `d(r, c)` — và chỉ thêm một lần (hai vòng `for` duyệt mỗi ô đúng một lần). ⇒ output chứa đủ `rows · cols` ô, không trùng.
- `max_d` là khoảng cách lớn nhất có thể từ tâm (đi tới góc xa nhất), nên chỉ số thùng luôn hợp lệ.
- Nối thùng theo `d` tăng dần ⇒ mọi ô ở thùng `d` đứng trước mọi ô ở thùng `d' > d` ⇒ dãy khoảng cách **không giảm**.

**BFS:** trên lưới không vật cản, số bước ít nhất từ tâm tới `(r, c)` bằng `|Δr| + |Δc|` (mỗi bước đổi đúng một tọa độ 1 đơn vị, và luôn có đường đi "thẳng" dài đúng bằng vậy). BFS lấy ô ra theo thứ tự số bước không giảm ⇒ đúng yêu cầu.

**Hình thoi:** với mỗi `d`, vòng `dr` và hai giá trị `dc = ±(d - |dr|)` liệt kê đúng các điểm nguyên có `|dr| + |dc| = d`; lọc theo biên ma trận ⇒ đúng tập ô có khoảng cách `d`.

## 9. Phân tích Time / Space

Gọi `N = rows · cols` (≤ 10 000), `D = max khoảng cách` (≤ 198).

| Cách | Ý tưởng | Thời gian | Bộ nhớ thêm |
|---|---|---|---|
| 1. Sinh + sort | so sánh khóa | `O(N log N)` | `O(N)` (Timsort) |
| 2. Bucket sort | thùng theo `d` | `O(N + D)` | `O(N + D)` |
| 3. BFS | sóng lan từ tâm | `O(N)` | `O(N)` (`visited` + queue) |
| 4. Viền hình thoi | duyệt `d`, `dr` | `O(D²)` = `O(N)`-ish* | `O(1)` ngoài output |

\* Cách 4 duyệt `Σ (2d + 1)` cặp `(d, dr)` với `d ≤ D` → `O(D²)`; vì `D ≤ rows + cols` nên `D² ≤ (rows + cols)²`, cùng bậc với `N` khi ma trận gần vuông, nhưng có thể lãng phí khi ma trận rất dẹt (ví dụ 1 × 100).

Output luôn có `N` ô → không thuật toán nào nhanh hơn `O(N)`.

## 10. Kiểm thử cho bài có **nhiều đáp án đúng**

Không thể `assert output == expected` vì thứ tự các ô cùng khoảng cách tùy ý. Thay vào đó kiểm tra **hai tính chất**:
1. Output là **hoán vị** của tập mọi ô (đủ, không trùng, không thừa).
2. Dãy khoảng cách **không giảm**.

In [18]:
def is_valid_answer(output, rows, cols, rCenter, cCenter) -> bool:
    all_cells = sorted([r, c] for r in range(rows) for c in range(cols))
    if sorted(output) != all_cells:
        return False
    dists = [abs(r - rCenter) + abs(c - cCenter) for r, c in output]
    return all(dists[i] <= dists[i + 1] for i in range(len(dists) - 1))

# Ba ví dụ của đề, kể cả các đáp án thay thế mà đề nói là hợp lệ
assert is_valid_answer([[0, 0], [0, 1]], 1, 2, 0, 0)
assert is_valid_answer([[0, 1], [0, 0], [1, 1], [1, 0]], 2, 2, 0, 1)
assert is_valid_answer([[0, 1], [1, 1], [0, 0], [1, 0]], 2, 2, 0, 1)
assert is_valid_answer([[1, 2], [0, 2], [1, 1], [0, 1], [1, 0], [0, 0]], 2, 3, 1, 2)
assert is_valid_answer([[1, 2], [1, 1], [0, 2], [1, 0], [0, 1], [0, 0]], 2, 3, 1, 2)
# Phản ví dụ: sai thứ tự, thiếu ô, trùng ô
assert not is_valid_answer([[0, 0], [0, 1], [1, 1], [1, 0]], 2, 2, 0, 1)
assert not is_valid_answer([[0, 1], [0, 0], [1, 1]], 2, 2, 0, 1)
assert not is_valid_answer([[0, 1], [0, 0], [0, 0], [1, 0]], 2, 2, 0, 1)
print("Bộ kiểm tra is_valid_answer hoạt động đúng")

Bộ kiểm tra is_valid_answer hoạt động đúng


Chú ý ta cũng test **bộ kiểm tra** bằng các phản ví dụ — nếu bộ kiểm tra sai, mọi test phía sau đều vô nghĩa.

In [19]:
import random
random.seed(1030)

solvers = {
    "sort": all_cells_sort,
    "sort_inplace": all_cells_sort_inplace,
    "bucket": all_cells_bucket,
    "bfs": all_cells_bfs,
    "diamond": all_cells_diamond,
    "Solution": Solution().allCellsDistOrder,
    "SolutionSort": SolutionSort().allCellsDistOrder,
}
fixed = [(1, 2, 0, 0), (2, 2, 0, 1), (2, 3, 1, 2), (1, 1, 0, 0), (1, 100, 0, 57), (100, 1, 99, 0), (100, 100, 0, 0), (100, 100, 50, 50)]
randoms = [(r, c, random.randrange(r), random.randrange(c))
           for r, c in [(random.randint(1, 15), random.randint(1, 15)) for _ in range(300)]]
for args in fixed + randoms:
    for name, f in solvers.items():
        out = f(*args)
        assert is_valid_answer(out, *args), (name, args)
print(f"{len(fixed) + len(randoms)} bộ tham số × {len(solvers)} cách: tất cả hợp lệ")

308 bộ tham số × 7 cách: tất cả hợp lệ


`f(*args)` **giải nén** tuple `args` thành các tham số riêng: `f(*(2, 3, 1, 2))` = `f(2, 3, 1, 2)`.

`random.randrange(r)` trả số nguyên trong `0..r-1` — đúng miền của `rCenter`.

## 11. Sai lầm người mới hay mắc

### 11.1. `[[]] * k` cho bucket

In [20]:
def wrong_shared_buckets(rows, cols, rCenter, cCenter):
    max_d = rows + cols
    buckets = [[]] * (max_d + 1)                 # SAI: mọi thùng là cùng một list
    for r in range(rows):
        for c in range(cols):
            buckets[abs(r - rCenter) + abs(c - cCenter)].append([r, c])
    result = []
    for bucket in buckets:
        result.extend(bucket)
    return result

out = wrong_shared_buckets(2, 2, 0, 1)
print("số ô trả về:", len(out), "(đúng phải là 4)")
print("hợp lệ?", is_valid_answer(out, 2, 2, 0, 1))

số ô trả về: 20 (đúng phải là 4)
hợp lệ? False


**Vì sao sai:** cả `max_d + 1` thùng cùng trỏ một list → `extend` lặp lại cùng nội dung nhiều lần, và thứ tự không theo khoảng cách. **Sửa:** `[[] for _ in range(k)]`.

### 11.2. Dùng khoảng cách Euclid thay vì Manhattan

In [21]:
def wrong_euclid(rows, cols, rCenter, cCenter):
    cells = [[r, c] for r in range(rows) for c in range(cols)]
    return sorted(cells, key=lambda rc: ((rc[0] - rCenter) ** 2 + (rc[1] - cCenter) ** 2) ** 0.5)

out = wrong_euclid(3, 3, 0, 0)
print(out)
print("hợp lệ theo Manhattan?", is_valid_answer(out, 3, 3, 0, 0))

[[0, 0], [0, 1], [1, 0], [1, 1], [0, 2], [2, 0], [1, 2], [2, 1], [2, 2]]
hợp lệ theo Manhattan? True


**Vì sao sai:** với lưới 3×3 tâm `(0,0)` ở trên, thứ tự Euclid **tình cờ** vẫn hợp lệ — đây là kiểu lỗi nguy hiểm vì test nhỏ không bắt được. Cell dưới tự tìm một input làm nó sai. **Sửa:** đọc kỹ định nghĩa trong đề: `|r1 - r2| + |c1 - c2|`.

In [22]:
found = None
for rows in range(1, 6):
    for cols in range(1, 6):
        for rc in range(rows):
            for cc in range(cols):
                if not is_valid_answer(wrong_euclid(rows, cols, rc, cc), rows, cols, rc, cc):
                    found = (rows, cols, rc, cc)
                    break
            if found: break
        if found: break
    if found: break
print("Input làm cách Euclid sai:", found)
out = wrong_euclid(*found)
print("Khoảng cách Manhattan theo thứ tự Euclid:", [abs(r - found[2]) + abs(c - found[3]) for r, c in out])

Input làm cách Euclid sai: (3, 4, 0, 0)
Khoảng cách Manhattan theo thứ tự Euclid: [0, 1, 1, 2, 2, 2, 3, 3, 4, 3, 4, 5]


Dãy khoảng cách Manhattan **bị giảm** ở một chỗ (`4` rồi `3`): Euclid xếp `(2,2)` (√8 ≈ 2.83) trước `(0,3)` (= 3), nhưng Manhattan của `(2,2)` là 4 > 3 → không hợp lệ.

### 11.3. Thiếu kiểm tra biên khi BFS / duyệt hình thoi

In [23]:
def wrong_no_bounds_diamond(rows, cols, rCenter, cCenter):
    max_d = max(rCenter, rows - 1 - rCenter) + max(cCenter, cols - 1 - cCenter)
    result = []
    for d in range(max_d + 1):
        for dr in range(-d, d + 1):
            k = d - abs(dr)
            for c in {cCenter - k, cCenter + k}:
                result.append([rCenter + dr, c])          # SAI: không lọc ô ngoài ma trận
    return result

out = wrong_no_bounds_diamond(2, 2, 0, 1)
print("vài ô trả về:", out[:6], "...")
print("hợp lệ?", is_valid_answer(out, 2, 2, 0, 1))

vài ô trả về: [[0, 1], [-1, 1], [0, 0], [0, 2], [1, 1], [-2, 1]] ...
hợp lệ? False


**Vì sao sai:** hình thoi tràn ra ngoài ma trận → có tọa độ âm hoặc ≥ `rows`/`cols`. **Sửa:** `if 0 <= r < rows` và `if 0 <= c < cols`.

### 11.4. Đánh dấu `visited` khi lấy ra (không phải khi đẩy vào)

Một ô có thể được đẩy vào queue nhiều lần (từ nhiều hàng xóm) trước khi được lấy ra → output có ô **trùng**. **Sửa:** đánh dấu ngay khi đẩy.

In [24]:
def wrong_mark_on_pop(rows, cols, rCenter, cCenter):
    visited = [[False] * cols for _ in range(rows)]
    queue = deque([(rCenter, cCenter)])
    result = []
    while queue:
        r, c = queue.popleft()
        visited[r][c] = True                     # SAI: đánh dấu muộn
        result.append([r, c])
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols and not visited[nr][nc]:
                queue.append((nr, nc))
    return result

out = wrong_mark_on_pop(2, 2, 0, 0)
print(out)
print("số ô:", len(out), "| hợp lệ?", is_valid_answer(out, 2, 2, 0, 0))

[[0, 0], [1, 0], [0, 1], [1, 1], [1, 1]]
số ô: 5 | hợp lệ? False


## 12. Nhận diện pattern

Dấu hiệu:
- "sắp xếp theo một khóa số nguyên trong miền nhỏ" → **bucket/counting sort** O(N + K).
- "khoảng cách trên lưới, đi 4 hướng, không vật cản" → khoảng cách Manhattan; có vật cản → **BFS**.
- "đáp án nào cũng được miễn thỏa điều kiện" → viết **hàm kiểm tra tính chất** thay vì so sánh cứng.

Bài liên quan:
- LeetCode 973 (K Closest Points to Origin) — khoảng cách Euclid, heap/quickselect.
- LeetCode 542 (01 Matrix), 994 (Rotting Oranges) — BFS nhiều nguồn trên lưới.
- LeetCode 1162 (As Far from Land as Possible) — BFS + Manhattan.
- LeetCode 451 (Sort Characters By Frequency), 347 (Top K Frequent Elements) — bucket sort theo tần suất.

## 13. Biến thể

### 13.1. Biến thể A — khoảng cách **Chebyshev** `max(|Δr|, |Δc|)` (đi được cả 8 hướng như quân vua)

In [25]:
def all_cells_chebyshev(rows, cols, rCenter, cCenter):
    max_d = max(rCenter, rows - 1 - rCenter, cCenter, cols - 1 - cCenter)
    buckets = [[] for _ in range(max_d + 1)]
    for r in range(rows):
        for c in range(cols):
            buckets[max(abs(r - rCenter), abs(c - cCenter))].append([r, c])
    return [cell for bucket in buckets for cell in bucket]

rows, cols, rc, cc = 5, 7, 2, 3
grid = [[0] * cols for _ in range(rows)]
for r, c in all_cells_chebyshev(rows, cols, rc, cc):
    grid[r][c] = max(abs(r - rc), abs(c - cc))
for row in grid:
    print(" ".join(map(str, row)))

3 2 2 2 2 2 3
3 2 1 1 1 2 3
3 2 1 0 1 2 3
3 2 1 1 1 2 3
3 2 2 2 2 2 3


Các ô cùng khoảng cách Chebyshev tạo thành **hình vuông** (so với hình thoi của Manhattan).

`[cell for bucket in buckets for cell in bucket]` là cách "làm phẳng" list các list bằng comprehension lồng.

### 13.2. Biến thể B — chỉ lấy các ô ở **đúng** khoảng cách `d`

In [26]:
def cells_at_distance(rows, cols, rCenter, cCenter, d):
    out = []
    for dr in range(-d, d + 1):
        r = rCenter + dr
        if 0 <= r < rows:
            k = d - abs(dr)
            for c in sorted({cCenter - k, cCenter + k}):
                if 0 <= c < cols:
                    out.append([r, c])
    return out

print(cells_at_distance(5, 7, 2, 3, 2))

[[0, 3], [1, 2], [1, 4], [2, 1], [2, 5], [3, 2], [3, 4], [4, 3]]


## 14. Hệ thống luyện tập — 6 bài

### Bài 1 — Tính tay
Với `rows = 3, cols = 3`, tâm `(0, 0)`: viết bảng khoảng cách 3×3 và một output hợp lệ.

### Bài 2 — Dự đoán output
Đoán `len(all_cells_bucket(4, 5, 3, 0))`, khoảng cách lớn nhất trong output đó, và `len(cells_at_distance(4, 5, 3, 0, 2))`.

### Bài 3 — TODO: viết lại bucket sort

In [27]:
def exercise_all_cells(rows: int, cols: int, rCenter: int, cCenter: int) -> list[list[int]]:
    # TODO 1: tính max_d
    # TODO 2: buckets = [[] for _ in range(max_d + 1)]
    # TODO 3: rải ô, rồi nối các thùng
    pass

# assert is_valid_answer(exercise_all_cells(2, 3, 1, 2), 2, 3, 1, 2)
# assert is_valid_answer(exercise_all_cells(7, 4, 3, 0), 7, 4, 3, 0)

### Bài 4 — TODO: đếm số ô ở mỗi khoảng cách (trả về list `count[d]`) — không tạo danh sách ô

In [28]:
def exercise_count_per_distance(rows: int, cols: int, rCenter: int, cCenter: int) -> list[int]:
    # TODO
    pass

# assert exercise_count_per_distance(2, 3, 1, 2) == [1, 2, 2, 1]
# assert exercise_count_per_distance(3, 3, 1, 1) == [1, 4, 4]

### Bài 5 — TODO biến thể: `k` ô gần tâm nhất (nếu hòa, ưu tiên hàng nhỏ hơn, rồi cột nhỏ hơn)

In [29]:
def exercise_k_nearest(rows: int, cols: int, rCenter: int, cCenter: int, k: int) -> list[list[int]]:
    # TODO
    pass

# assert exercise_k_nearest(3, 3, 1, 1, 3) == [[1, 1], [0, 1], [1, 0]]
# assert exercise_k_nearest(2, 3, 1, 2, 4) == [[1, 2], [0, 2], [1, 1], [0, 1]]

### Bài 6 — Mini challenge: thứ tự theo khoảng cách từ **nhiều tâm** (lấy khoảng cách tới tâm gần nhất)

Cho danh sách tâm `centers`, trả về mọi ô sắp theo `min` khoảng cách Manhattan tới các tâm. Gợi ý: BFS **nhiều nguồn**.

In [30]:
def exercise_multi_center(rows: int, cols: int, centers: list[list[int]]) -> list[list[int]]:
    # TODO
    pass

# out = exercise_multi_center(3, 4, [[0, 0], [2, 3]])
# d = lambda r, c: min(abs(r - a) + abs(c - b2) for a, b2 in [[0, 0], [2, 3]])
# assert sorted(out) == sorted([r, c] for r in range(3) for c in range(4))
# assert all(d(*out[i]) <= d(*out[i + 1]) for i in range(len(out) - 1))

## 15. Gợi ý ba mức

**Bài 1** — (1) `d = r + c` vì tâm ở góc; (2) bảng có `0..4`; (3) thứ tự theo đường chéo phụ.

**Bài 2** — (1) `N = rows · cols`; (2) góc xa nhất từ `(3, 0)` là `(0, 4)`; (3) liệt kê `dr = -2..0` (vì `r ≤ 3`).

**Bài 3** — (1) mục 5; (2) `buckets[d].append([r, c])`; (3) `result.extend(bucket)`.

**Bài 4** — (1) giống bucket nhưng chỉ đếm; (2) `count = [0] * (max_d + 1)`; (3) `count[d] += 1`.

**Bài 5** — (1) sort với khóa là **tuple** `(d, r, c)` — Python so sánh tuple từng phần tử; (2) lấy `[:k]`; (3) `key=lambda rc: (abs(...)+abs(...), rc[0], rc[1])`.

**Bài 6** — (1) đẩy **tất cả** tâm vào queue lúc đầu, đánh dấu visited; (2) BFS như mục 5.1; (3) thứ tự lấy ra là đáp án.

## 16. Đáp án tham khảo

**Bài 1:**
```
0 1 2
1 2 3
2 3 4
```
Một output hợp lệ: `[[0,0],[0,1],[1,0],[0,2],[1,1],[2,0],[1,2],[2,1],[2,2]]`.

**Bài 2:** `20`; khoảng cách lớn nhất `3 + 4 = 7`; `len(cells_at_distance(4,5,3,0,2)) = 3` (`(1,0)`, `(2,1)`, `(3,2)`).

In [31]:
def answer_all_cells(rows, cols, rCenter, cCenter):
    max_d = max(rCenter, rows - 1 - rCenter) + max(cCenter, cols - 1 - cCenter)
    buckets = [[] for _ in range(max_d + 1)]
    for r in range(rows):
        for c in range(cols):
            buckets[abs(r - rCenter) + abs(c - cCenter)].append([r, c])
    result = []
    for bucket in buckets:
        result.extend(bucket)
    return result


def answer_count_per_distance(rows, cols, rCenter, cCenter):
    max_d = max(rCenter, rows - 1 - rCenter) + max(cCenter, cols - 1 - cCenter)
    count = [0] * (max_d + 1)
    for r in range(rows):
        for c in range(cols):
            count[abs(r - rCenter) + abs(c - cCenter)] += 1
    return count


def answer_k_nearest(rows, cols, rCenter, cCenter, k):
    cells = [[r, c] for r in range(rows) for c in range(cols)]
    cells.sort(key=lambda rc: (abs(rc[0] - rCenter) + abs(rc[1] - cCenter), rc[0], rc[1]))
    return cells[:k]


def answer_multi_center(rows, cols, centers):
    visited = [[False] * cols for _ in range(rows)]
    queue = deque()
    for r, c in centers:
        if not visited[r][c]:
            visited[r][c] = True
            queue.append((r, c))
    result = []
    while queue:
        r, c = queue.popleft()
        result.append([r, c])
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols and not visited[nr][nc]:
                visited[nr][nc] = True
                queue.append((nr, nc))
    return result


assert is_valid_answer([[0,0],[0,1],[1,0],[0,2],[1,1],[2,0],[1,2],[2,1],[2,2]], 3, 3, 0, 0)
out = all_cells_bucket(4, 5, 3, 0)
assert len(out) == 20 and max(abs(r - 3) + abs(c - 0) for r, c in out) == 7
assert cells_at_distance(4, 5, 3, 0, 2) == [[1, 0], [2, 1], [3, 2]]
assert is_valid_answer(answer_all_cells(2, 3, 1, 2), 2, 3, 1, 2)
assert is_valid_answer(answer_all_cells(7, 4, 3, 0), 7, 4, 3, 0)
assert answer_count_per_distance(2, 3, 1, 2) == [1, 2, 2, 1]
assert answer_count_per_distance(3, 3, 1, 1) == [1, 4, 4]
assert answer_k_nearest(3, 3, 1, 1, 3) == [[1, 1], [0, 1], [1, 0]]
assert answer_k_nearest(2, 3, 1, 2, 4) == [[1, 2], [0, 2], [1, 1], [0, 1]]

random.seed(77)
for _ in range(200):
    rows, cols = random.randint(1, 10), random.randint(1, 10)
    centers = [[random.randrange(rows), random.randrange(cols)] for _ in range(random.randint(1, 3))]
    out = answer_multi_center(rows, cols, centers)
    dist = lambda r, c: min(abs(r - a) + abs(c - b2) for a, b2 in centers)
    assert sorted(out) == sorted([r, c] for r in range(rows) for c in range(cols))
    assert all(dist(*out[i]) <= dist(*out[i + 1]) for i in range(len(out) - 1))
    rc, cc = centers[0]
    assert answer_count_per_distance(rows, cols, rc, cc) == [len(cells_at_distance(rows, cols, rc, cc, d))
                                                           for d in range(len(answer_count_per_distance(rows, cols, rc, cc)))]
print("Đáp án tham khảo PASS toàn bộ test!")

Đáp án tham khảo PASS toàn bộ test!


**Giải thích bài 5:** khóa là tuple `(d, r, c)`. Python so sánh tuple theo **thứ tự từ điển**: so `d` trước; nếu bằng thì so `r`; nếu vẫn bằng thì so `c`. Nhờ vậy ta có thứ tự xác định duy nhất, dễ test bằng `==`.

**Bài 6:** BFS nhiều nguồn giống thả nhiều viên sỏi xuống nước cùng lúc: sóng lan đồng thời từ mọi tâm, mỗi ô được chạm lần đầu bởi tâm gần nhất.

## 17. Tổng kết — template + checklist

**Template nhớ nhanh (bucket sort theo khoảng cách):**

```python
class Solution:
    def allCellsDistOrder(self, rows, cols, rCenter, cCenter):
        max_d = max(rCenter, rows - 1 - rCenter) + max(cCenter, cols - 1 - cCenter)
        buckets = [[] for _ in range(max_d + 1)]
        for r in range(rows):
            for c in range(cols):
                buckets[abs(r - rCenter) + abs(c - cCenter)].append([r, c])
        return [cell for bucket in buckets for cell in bucket]
```

**Câu thần chú:** *"Khóa nhỏ, nguyên → bỏ vào thùng. Lưới 4 hướng → Manhattan = số bước BFS."*

**Checklist trước khi submit:**
- [ ] Khoảng cách là `abs(Δr) + abs(Δc)` (Manhattan, không phải Euclid)?
- [ ] Bucket tạo bằng `[[] for _ in range(k)]` (không phải `[[]] * k`)?
- [ ] Số thùng đủ cho khoảng cách lớn nhất?
- [ ] Trả về `list[list[int]]` (mỗi ô là `[r, c]`)?
- [ ] Nếu dùng BFS: đánh dấu `visited` khi **đẩy vào** queue, có kiểm tra biên?
- [ ] Test bằng **hàm kiểm tra tính chất**, không so sánh cứng với một thứ tự?